In [35]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tvDatafeed import TvDatafeed, Interval
from ta.trend import MACD



In [116]:


# Login no TradingView
tv = TvDatafeed()

ticker = 'BTCUSD'
exchange = 'INDEX'


df = tv.get_hist(
    symbol=ticker,
    exchange=exchange,
    interval=Interval.in_daily,
    n_bars=10000
)
df = df[df.index.year >=2012].dropna()
df.index = pd.to_datetime(df.index).normalize().date
df.drop(columns='symbol',inplace=True)
df.index = pd.to_datetime(df.index).normalize()
df.dropna(inplace=True)
df['ret'] = df['close'].pct_change()
df.tail()




,open,high,low,close,volume,ret
2025-12-19,88136.46,88564.48,87822.75,88360.53,3434.908996,0.002576
2025-12-20,88361.78,89062.34,87598.20,88656.04,4822.030773,0.003344
2025-12-21,88661.39,90559.10,87881.69,88573.74,15543.350693,-0.000928
2025-12-22,88578.13,88912.61,86573.04,87442.93,12331.202633,-0.012767
2025-12-23,87441.43,87786.25,87221.74,87576.75,600.173211,0.001530


In [117]:
macd_ind = MACD(
    close=df["close"],
    window_fast=12,
    window_slow=26,
    window_sign=9
)

df["macd"] = macd_ind.macd()
df["macd_signal"] = macd_ind.macd_signal()
df['macd_hist'] = macd_ind.macd_diff()


df["buy_signal"] = (
    (df["macd"] > df["macd_signal"]) &
    (df["macd"].shift(1) <= df["macd_signal"].shift(1))
)

df["sell_signal"] = (
    (df["macd"] < df["macd_signal"]) &
    (df["macd"].shift(1) >= df["macd_signal"].shift(1))
)

df['position'] = np.where(df["macd"] > df["macd_signal"], 1, 0)
df["strategy_ret"] = df["position"].shift(1) * df["ret"]


df["strategy"] = df["strategy_ret"].cumsum()
df["buy_hold"] = df["ret"].cumsum()
df.tail()

,open,high,low,close,volume,ret,macd,macd_signal,macd_hist,buy_signal,sell_signal,position,strategy_ret,strategy,buy_hold
2025-12-19,88136.46,88564.48,87822.75,88360.53,3434.908996,0.002576,-1710.080263,-1775.240730,65.160466,True,False,1,0.000000,12.220562,13.985191
2025-12-20,88361.78,89062.34,87598.20,88656.04,4822.030773,0.003344,-1567.433076,-1733.679199,166.246123,False,False,1,0.003344,12.223907,13.988535
2025-12-21,88661.39,90559.10,87881.69,88573.74,15543.350693,-0.000928,-1444.375216,-1675.818402,231.443186,False,False,1,-0.000928,12.222979,13.987607
2025-12-22,88578.13,88912.61,86573.04,87442.93,12331.202633,-0.012767,-1421.709312,-1624.996584,203.287272,False,False,1,-0.012767,12.210212,13.974840
2025-12-23,87441.43,87786.25,87221.74,87576.75,600.173211,0.001530,-1377.074210,-1575.412109,198.337900,False,False,1,0.001530,12.211742,13.976371


In [118]:
df_plot = df.tail(500)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=(f"{ticker} Close Price", "MACD"))

# Close price
fig.add_trace(
    go.Scatter(x=df_plot.index, y=df_plot["close"], name="Close Price", line=dict(color="black")),
    row=1, col=1
)
# Buy / Sell markers on price
fig.add_trace(
    go.Scatter(x=df_plot.index[df_plot["buy_signal"]],
               y=df_plot.loc[df_plot["buy_signal"], "close"],
               mode="markers", name="Start Bullish Run",
               marker=dict(symbol="triangle-up", color="green", size=10)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df_plot.index[df_plot["sell_signal"]],
               y=df_plot.loc[df_plot["sell_signal"], "close"],
               mode="markers", name="Start Bearish Run",
               marker=dict(symbol="triangle-down", color="red", size=10)),
    row=1, col=1
)

# MACD lines and histogram
fig.add_trace(
    go.Scatter(x=df_plot.index, y=df_plot["macd"], name="MACD", line=dict(color="blue")),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df_plot.index, y=df_plot["macd_signal"], name="Signal", line=dict(color="red")),
    row=2, col=1
)
fig.add_trace(
    go.Bar(x=df_plot.index, y=df_plot["macd_hist"], name="Histogram", marker_color="grey", opacity=0.6),
    row=2, col=1
)
# Buy / Sell markers on MACD
fig.add_trace(
    go.Scatter(x=df_plot.index[df_plot["buy_signal"]],
               y=df_plot.loc[df_plot["buy_signal"], "macd"],
               mode="markers", name="Bullish Crossover",
               marker=dict(symbol="triangle-up", color="green", size=10)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df_plot.index[df_plot["sell_signal"]],
               y=df_plot.loc[df_plot["sell_signal"], "macd"],
               mode="markers", name="Bearish Crossover",
               marker=dict(symbol="triangle-down", color="red", size=10)),
    row=2, col=1
)

fig.update_layout(title_text=f"{ticker} | Close Price and MACD", height=700, template="plotly_white",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="MACD", row=2, col=1)
fig.show()

In [123]:
# =========================
# PLOT (styled to match previous Plotly figure)
# =========================
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.65, 0.25],
    subplot_titles=(f"{ticker} | Cumulative Returns", "Position")
)

# Buy & Hold
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["buy_hold"] * 100,
        name="Buy & Hold",
        line=dict(width=2, color="black")
    ),
    row=1, col=1
)

# Strategy
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["strategy"] * 100,
        name="Strategy",
        line=dict(width=2, color="blue")
    ),
    row=1, col=1
)

# Position (filled area)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["position"],
        name="Position",
        mode="lines",
        line=dict(width=1, color="green"),
        fill="tozeroy",
        fillcolor="rgba(0,200,0,0.08)"
    ),
    row=2, col=1
)

# Layout
fig.update_layout(
    title_text=f"{ticker} | MACD",
    template="plotly_white",
    height=700,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_yaxes(title_text="Cumulative Return (%)", row=1, col=1)
fig.update_yaxes(title_text="Position", row=2, col=1, range=[-0.05, 1.05])
fig.update_xaxes(showticklabels=False, row=1, col=1)

fig.show()




In [120]:
total_return_bh = df["buy_hold"].iloc[-1]
total_return_strategy = df["strategy"].iloc[-1]

vol_strategy = df["strategy_ret"].std() * np.sqrt(252)
vol_bh = df["ret"].std() * np.sqrt(252)

sharpe_strategy = (
    df["strategy_ret"].mean() / df["strategy_ret"].std()
) * np.sqrt(252)

sharpe_bh = (
    df["ret"].mean() / df["ret"].std()
) * np.sqrt(252)

print("=== RESULTADOS ===")
print(f"Buy & Hold Return: {total_return_bh:.2%}")
print(f"Strategy Return:   {total_return_strategy:.2%}")
print()
print(f"Buy & Hold Vol: {vol_bh:.2%}")
print(f"Strategy Vol:   {vol_strategy:.2%}")
print()
print(f"Buy & Hold Sharpe: {sharpe_bh:.2f}")
print(f"Strategy Sharpe:   {sharpe_strategy:.2f}")


=== RESULTADOS ===
Buy & Hold Return: 1397.64%
Strategy Return:   1221.17%

Buy & Hold Vol: 64.68%
Strategy Vol:   44.74%

Buy & Hold Sharpe: 1.07
Strategy Sharpe:   1.35
